In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="07-event-recommendation",
    notebook_name="02_location_and_time_features.ipynb"
)

# Location, Time, Social, and Cold-Start Features

## For a 12-Year-Old

Imagine you want to go to a concert this Saturday. Here is what matters:

- **Location**: Is it close enough? Can I walk there, or do I need a car?
- **Time**: Is it on a day I am free? Is there enough time to get ready?
- **Friends**: Are my friends going? Did someone invite me?
- **New events**: A brand-new event was just posted 5 minutes ago -- how do we recommend it when nobody has tried it yet?

This notebook dives deep into each of these challenges.

---

## Staff-Level Summary

Event recommendation has unique challenges compared to standard content recommendation:
1. **Geospatial features** -- physical accessibility dominates user decisions
2. **Temporal dynamics** -- events expire; time-to-event and user schedule preferences matter
3. **Social signals** -- friend attendance, invitations, and host relationships are strong indicators
4. **Constant cold start** -- events are ephemeral; most have near-zero interaction history

## 1. Geospatial Features and Location-Based Filtering

### Why Location Is Everything

For Netflix, location does not matter -- you stream from anywhere. But for events, a concert in Tokyo is completely useless to someone in Chicago. Location is the single biggest filter.

### The Three Location Questions

1. **How accessible is the event's location?** (Walk score, transit score, bike score)
2. **Is the event in my city/country?** (Binary features)
3. **How far is it from me, and am I comfortable with that distance?** (Distance + similarity)

In [ ]:
import numpy as np
import pandas as pd
from math import radians, cos, sin, asin, sqrt

# === Walk Score System ===
# Walk score is a number between 0-100 that measures how walkable an address is.
# It comes from external APIs (Google Maps, OpenStreetMap, WalkScore.com).

walk_score_categories = {
    1: {"range": "90-100", "description": "No car needed", "example": "Downtown Manhattan"},
    2: {"range": "70-89",  "description": "Very walkable",  "example": "Most urban neighborhoods"},
    3: {"range": "50-69",  "description": "Somewhat walkable", "example": "Suburban shopping areas"},
    4: {"range": "25-49",  "description": "Car-dependent",  "example": "Typical suburbs"},
    5: {"range": "0-24",   "description": "Requires a car", "example": "Rural areas, remote venues"}
}

print("=== Walk Score Categories ===")
print(f"{'Category':>10} | {'Score Range':>12} | {'Description':<20} | {'Example'}")
print("-" * 75)
for cat, info in walk_score_categories.items():
    print(f"{cat:>10} | {info['range']:>12} | {info['description']:<20} | {info['example']}")

print("\n12-year-old version:")
print("  Walk score = 95 means 'You can walk to everything around the venue.'")
print("  Walk score = 15 means 'You definitely need a car to get there.'")

In [ ]:
# === Walk Score SIMILARITY Feature ===
# This is a key insight: it is not just about the event's walk score,
# but how it compares to the user's HISTORICAL PREFERENCE.

def compute_walk_score_similarity(event_walk_score, user_past_walk_scores):
    """
    How different is this event's walkability from what the user usually prefers?
    
    12-year-old version:
    If you always go to events in walkable areas (score ~90),
    and this event has a walk score of 20 (middle of nowhere),
    you probably will not like it. The SIMILARITY tells us this.
    
    Staff-level detail:
    We compute the difference between the candidate event's walk score
    and the user's average walk score across previously registered events.
    A small difference = good match. A large difference = bad match.
    """
    user_avg = np.mean(user_past_walk_scores)
    similarity = abs(event_walk_score - user_avg)
    return similarity, user_avg


# Example: User who loves walkable venues
user_past_walk_scores = [92, 88, 95, 90, 85]  # Loves walkable areas
event_walk_score_good = 91  # Very walkable event
event_walk_score_bad = 25   # Remote venue

sim_good, avg = compute_walk_score_similarity(event_walk_score_good, user_past_walk_scores)
sim_bad, _ = compute_walk_score_similarity(event_walk_score_bad, user_past_walk_scores)

print(f"User's average walk score: {avg:.0f}")
print(f"\nEvent A (walk score {event_walk_score_good}): similarity = {sim_good:.1f} (SMALL = good match)")
print(f"Event B (walk score {event_walk_score_bad}): similarity = {sim_bad:.1f} (LARGE = bad match)")
print(f"\nThe model learns that smaller similarity values -> more likely to register")

In [ ]:
# === Distance Features ===

def haversine(lat1, lon1, lat2, lon2):
    """Calculate distance between two points in miles."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return 3956 * c  # Earth radius in miles


def bucketize_distance(distance_miles):
    """Convert distance to category.
    
    12-year-old version:
    0 = 'right around the corner' (< 1 mile)
    1 = 'a short bike ride' (1-5 miles)
    2 = 'need a car ride' (5-20 miles)
    3 = 'a bit of a drive' (20-50 miles)
    4 = 'a road trip' (50-100 miles)
    5 = 'need to fly or take a long drive' (100+ miles)
    """
    thresholds = [1, 5, 20, 50, 100]
    for i, t in enumerate(thresholds):
        if distance_miles < t:
            return i
    return 5


def compute_distance_similarity(event_distance, user_past_distances):
    """
    How different is this event's distance from the user's comfort zone?
    
    Some users are willing to travel 50 miles for a concert.
    Others will only go to events within 5 miles.
    This feature captures that personal preference.
    """
    user_avg = np.mean(user_past_distances)
    return abs(event_distance - user_avg), user_avg


# Simulate: User in Miami with 5 past events at various distances
user_lat, user_lon = 25.7617, -80.1918  # Miami

events_locations = {
    "Dua Lipa (Miami Beach)": (25.7903, -80.1303),
    "Warriors Game (NYC)": (40.7505, -73.9934),
    "Jazz Night (NYC)": (40.7306, -74.0003),
    "Wine Festival (Chicago)": (41.8827, -87.6233),
    "AI Meetup (Miami)": (25.7750, -80.1950),
}

print("=== Distance From Miami User to Each Event ===")
for name, (lat, lon) in events_locations.items():
    dist = haversine(user_lat, user_lon, lat, lon)
    bucket = bucketize_distance(dist)
    bucket_labels = ['<1mi', '1-5mi', '5-20mi', '20-50mi', '50-100mi', '100+mi']
    print(f"  {name:35s}: {dist:8.1f} miles (bucket {bucket}: {bucket_labels[bucket]})")

print("\nKey insight: distance bucket is one-hot encoded as a feature.")
print("Distance SIMILARITY compares to user's past events.")

In [ ]:
# === Location as Model Input: Efficiency Consideration ===

# IMPORTANT INTERVIEW TALKING POINT:
# Instead of computing distance as a feature (requires real-time API call),
# you can pass BOTH locations to the model and let IT learn.

print("=== Feature Computation Efficiency ===")
print()
print("Option A (Explicit feature): Compute distance via Google Maps API")
print("  Pros: Clear signal, human-interpretable")
print("  Cons: Requires API call per (user, event) pair = EXPENSIVE at scale")
print("  Cost: 500 candidates x 1M users = 500M API calls/day")
print()
print("Option B (Raw locations as features): Pass user_lat, user_lon, event_lat, event_lon")
print("  Pros: No API calls, model learns distance implicitly")
print("  Cons: Model must learn the distance function from data")
print("  Cost: Zero additional API calls")
print()
print("Option C (Hybrid): Pre-compute distances in batch, cache in feature store")
print("  Pros: Best of both worlds")
print("  Cons: Storage overhead, stale for new events")
print()
print(">>> In practice: Option C is preferred. Pre-compute for nearby events,")
print("    use Option B as fallback for new events.")

## 2. Time-Aware Recommendations (Temporal Dynamics)

### The Time Challenge

Events are not like products. A product sits on a shelf forever. An event:
- Starts at a specific time and ends
- Has a window of "remaining time" before it starts
- Different users prefer different times (some are weekend people, some are weeknight people)

### The Four Time Questions

1. **How much time is left until the event?** (Remaining time bucket)
2. **How long to get there?** (Estimated travel time)
3. **Is the day convenient?** (Day-of-week profile)
4. **Is the hour convenient?** (Hour-of-day profile)

In [ ]:
from datetime import datetime, timedelta

# === Remaining Time Feature ===

def bucketize_remaining_time(hours_remaining):
    """
    Bucketize time remaining until event starts.
    
    12-year-old version:
    0 = 'Starting any minute!' (< 1 hour)
    1 = 'Very soon' (1-2 hours)
    2 = 'This afternoon' (2-4 hours)
    3 = 'Later today' (4-6 hours)
    4 = 'Tonight' (6-12 hours)
    5 = 'Tomorrow' (12-24 hours)
    6 = 'In a few days' (1-3 days)
    7 = 'This week' (3-7 days)
    8 = 'Next week or later' (7+ days)
    """
    thresholds = [1, 2, 4, 6, 12, 24, 72, 168]
    for i, t in enumerate(thresholds):
        if hours_remaining < t:
            return i
    return 8


# Simulate "now" and various event times
now = datetime(2024, 3, 13, 14, 0)  # Wednesday 2pm

event_times = {
    "Jazz Night (tonight at 10pm)": datetime(2024, 3, 13, 22, 0),
    "Comedy Show (tomorrow 8pm)": datetime(2024, 3, 14, 20, 0),
    "Dua Lipa (Friday 8pm)": datetime(2024, 3, 15, 20, 0),
    "Basketball (Saturday 7pm)": datetime(2024, 3, 16, 19, 0),
    "Wine Festival (next Wed 6pm)": datetime(2024, 3, 20, 18, 0),
}

bucket_labels = [
    '<1hr (Starting!)', '1-2hr (Very soon)', '2-4hr (This afternoon)',
    '4-6hr (Later today)', '6-12hr (Tonight)', '12-24hr (Tomorrow)',
    '1-3 days', '3-7 days', '7+ days'
]

print(f"Current time: {now.strftime('%A %I:%M %p')}")
print(f"\n{'Event':<35} {'Hours Left':>10} {'Bucket':>8} Label")
print("-" * 80)
for name, event_time in event_times.items():
    hours = (event_time - now).total_seconds() / 3600
    bucket = bucketize_remaining_time(hours)
    print(f"{name:<35} {hours:>10.1f} {bucket:>8} {bucket_labels[bucket]}")

print("\nKey insight: Users have different planning horizons.")
print("Some plan a week ahead; others decide 2 hours before.")
print("The SIMILARITY feature captures this personal preference.")

In [ ]:
# === Day-of-Week and Hour-of-Day User Profiles ===

def build_temporal_profile(past_event_datetimes):
    """
    Build a user's temporal profile from past registered events.
    
    12-year-old version:
    By looking at when you went to events before, we build a 'schedule'
    that shows when you USUALLY go to events.
    - If you always go on Saturdays, your Saturday score is HIGH.
    - If you never go on Mondays, your Monday score is ZERO.
    
    Staff-level detail:
    We create two histograms:
    1. Day-of-week profile: 7-dimensional vector (Mon=0 through Sun=6)
    2. Hour-of-day profile: 24-dimensional vector (0-23)
    Each value is the RATE of attendance (count / total events).
    """
    day_counts = np.zeros(7)   # Mon through Sun
    hour_counts = np.zeros(24) # 0 through 23
    
    for dt in past_event_datetimes:
        day_counts[dt.weekday()] += 1
        hour_counts[dt.hour] += 1
    
    total = len(past_event_datetimes)
    if total > 0:
        day_profile = day_counts / total
        hour_profile = hour_counts / total
    else:
        day_profile = np.ones(7) / 7      # Uniform prior
        hour_profile = np.ones(24) / 24
    
    return day_profile, hour_profile


# Simulate a user's past event attendance
past_events = [
    datetime(2024, 1, 6, 20, 0),   # Saturday 8pm
    datetime(2024, 1, 13, 19, 0),  # Saturday 7pm
    datetime(2024, 1, 20, 21, 0),  # Saturday 9pm
    datetime(2024, 1, 26, 20, 0),  # Friday 8pm
    datetime(2024, 2, 2, 22, 0),   # Friday 10pm
    datetime(2024, 2, 3, 15, 0),   # Saturday 3pm
    datetime(2024, 2, 10, 20, 0),  # Saturday 8pm
    datetime(2024, 2, 14, 19, 0),  # Wednesday 7pm (Valentine's!)
    datetime(2024, 2, 17, 18, 0),  # Saturday 6pm
    datetime(2024, 3, 2, 20, 0),   # Saturday 8pm
]

day_profile, hour_profile = build_temporal_profile(past_events)

days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print("=== User's Day-of-Week Profile ===")
print("(Probability of attending events on each day)\n")
for i, (day, rate) in enumerate(zip(days, day_profile)):
    bar = '#' * int(rate * 50)
    print(f"  {day}: {rate:.2f} {bar}")

print(f"\nThis user is a WEEKEND person (Fri/Sat = {day_profile[4]+day_profile[5]:.0%} of events).")
print(f"Recommending a Monday event would be a bad idea.")

print("\n=== User's Hour-of-Day Profile ===")
print("(Showing only non-zero hours)\n")
for h in range(24):
    if hour_profile[h] > 0:
        bar = '#' * int(hour_profile[h] * 50)
        print(f"  {h:2d}:00: {hour_profile[h]:.2f} {bar}")

print(f"\nThis user prefers EVENING events (7-10pm).")

In [ ]:
# === Temporal Similarity Features ===

def compute_temporal_similarity(event_datetime, day_profile, hour_profile):
    """
    How well does the event's timing match the user's historical preferences?
    
    Returns:
    - day_match: user's historical rate for this day of week (higher = better match)
    - hour_match: user's historical rate for this hour (higher = better match)
    """
    day_match = day_profile[event_datetime.weekday()]
    hour_match = hour_profile[event_datetime.hour]
    return day_match, hour_match


# Test with different event times
test_events = {
    "Saturday 8pm Concert": datetime(2024, 3, 16, 20, 0),
    "Monday 9am Workshop": datetime(2024, 3, 18, 9, 0),
    "Friday 7pm Jazz": datetime(2024, 3, 22, 19, 0),
    "Wednesday 2pm Talk": datetime(2024, 3, 20, 14, 0),
}

print("=== Temporal Match Scores ===")
print(f"{'Event':<25} {'Day Match':>10} {'Hour Match':>12} {'Verdict'}")
print("-" * 70)
for name, dt in test_events.items():
    day_m, hour_m = compute_temporal_similarity(dt, day_profile, hour_profile)
    if day_m > 0.2 and hour_m > 0:
        verdict = "GREAT match"
    elif day_m > 0.1:
        verdict = "OK match"
    else:
        verdict = "POOR match"
    print(f"{name:<25} {day_m:>10.2f} {hour_m:>12.2f} {verdict}")

print("\nThe model uses these match scores as features to predict registration.")

## 3. Handling Ephemeral Events (They Expire!)

### The Core Challenge

This is what makes event recommendation fundamentally different from movie or product recommendation.

**Movies/Products:** Stay forever. Accumulate thousands of ratings. Collaborative filtering works great.

**Events:** Created, exist briefly, happen, and expire. Most events have very few interactions before they happen. This is a **constant cold-start problem**.

In [ ]:
# === The Event Lifecycle ===

def simulate_event_lifecycle():
    """
    Show the typical lifecycle of an event and why cold-start is constant.
    
    12-year-old version:
    Imagine a new concert just got posted. Nobody has seen it yet.
    How do we recommend it? We can not use 'users who liked this
    also liked that' because NOBODY has 'liked' it yet!
    
    The event only exists for maybe 2 weeks, gets a few hundred
    views, then it happens and disappears forever.
    """
    print("=== Typical Event Lifecycle ===")
    print()
    
    lifecycle = [
        ("Day 0", "Event created by host", 0, 0, "COLD START -- zero interactions"),
        ("Day 1", "First impressions shown", 50, 2, "Still very cold"),
        ("Day 3", "Some engagement", 200, 8, "Warming up, but still sparse"),
        ("Day 7", "Peak discovery period", 800, 30, "Some signal, but limited"),
        ("Day 10", "Approaching event date", 1200, 45, "Best signal we will ever have"),
        ("Day 14", "EVENT DAY", 1500, 50, "Too late to recommend!"),
        ("Day 15", "Event is OVER", 1500, 50, "EXPIRED -- no longer relevant"),
    ]
    
    print(f"{'Time':<10} {'Status':<30} {'Impressions':>12} {'Registrations':>15} Notes")
    print("-" * 100)
    for time, status, impressions, regs, notes in lifecycle:
        print(f"{time:<10} {status:<30} {impressions:>12} {regs:>15} {notes}")
    
    print("\n=== Compare to a Movie on Netflix ===")
    movie_lifecycle = [
        ("Month 1", "Movie added", 10000, 5000),
        ("Month 6", "Accumulating ratings", 100000, 50000),
        ("Year 1", "Well-established", 500000, 250000),
        ("Year 5", "Classic film", 2000000, 1000000),
    ]
    
    print(f"{'Time':<10} {'Status':<30} {'Views':>12} {'Ratings':>12}")
    print("-" * 70)
    for time, status, views, ratings in movie_lifecycle:
        print(f"{time:<10} {status:<30} {views:>12} {ratings:>12}")
    
    print("\nConclusion: Events have 100x-1000x FEWER interactions than movies.")
    print("This is why FEATURE ENGINEERING is so critical for event recommendation.")
    print("We cannot rely on collaborative filtering alone.")


simulate_event_lifecycle()

In [ ]:
# === Cold Start Strategies ===

def demonstrate_cold_start_strategies():
    """
    How to recommend events with zero or near-zero interaction history.
    
    12-year-old version:
    A brand-new event was just posted. We know NOTHING about who likes it.
    But we DO know:
    - Its category (music, sports, tech)
    - Its location (Miami, NYC)
    - Its price ($50)
    - Its description ('Jazz concert with live band')
    - Who created it (host with 50 past events)
    
    We use ALL of this to make predictions without needing anyone
    to have interacted with it yet!
    """
    strategies = {
        "1. Content-Based Features": {
            "how": "Use event attributes (category, price, location, description) to match user preferences",
            "example": "User likes Jazz + event is Jazz -> recommend even with 0 interactions",
            "key_features": ["Description similarity (TF-IDF cosine)", "Category match", "Price match"]
        },
        "2. Location & Time Features": {
            "how": "These features are available from the moment the event is created",
            "example": "Event in Miami + user in Miami + user prefers short distances -> recommend",
            "key_features": ["Distance", "Walk score", "Day/hour match"]
        },
        "3. Host-Based Features": {
            "how": "Transfer knowledge from the host's previous events",
            "example": "This host's past events got 80% registration -> good signal",
            "key_features": ["Host popularity", "Host's avg registration rate", "Host is friend?"]
        },
        "4. Social Features": {
            "how": "Early social signals even before many registrations",
            "example": "2 of your friends registered in the first hour -> strong signal",
            "key_features": ["Friends registered (even if count is low)", "Invitation from friends"]
        },
        "5. Similarity Features": {
            "how": "Compare event attributes to user's historical preferences",
            "example": "User's avg event price = $60, this event = $50 -> price similarity = 10",
            "key_features": ["Walk score sim", "Price sim", "Distance sim", "Remaining time sim"]
        }
    }
    
    print("=== Cold Start Strategies for New Events ===")
    for strategy, details in strategies.items():
        print(f"\n{strategy}")
        print(f"  How: {details['how']}")
        print(f"  Example: {details['example']}")
        print(f"  Key features:")
        for feat in details['key_features']:
            print(f"    - {feat}")
    
    print("\n=== Key Insight ===")
    print("Because ALL our features are computable from event attributes,")
    print("user profile, and social graph -- NOT from interaction history --")
    print("our model can recommend brand-new events from the moment they are created!")
    print("\nThis is why feature engineering is the STAR of event recommendation.")


demonstrate_cold_start_strategies()

## 4. Social Features (Friends Attending)

### Why Social Signals Matter

Would you rather go to a concert alone, or a concert where 5 of your friends are going? Social features are some of the **strongest predictive signals** in event recommendation.

In [ ]:
# === Complete Social Feature Pipeline ===

class SocialFeatureComputer:
    """
    Compute all social-related features for event recommendation.
    
    12-year-old version:
    These features answer questions like:
    - 'How many people signed up?' (popularity)
    - 'How many of MY FRIENDS signed up?' (social proof)
    - 'Did someone invite me?' (personal invitation)
    - 'Is the event host my friend?' (trust signal)
    - 'Did I go to this host's events before?' (loyalty)
    """
    
    def __init__(self, friendships_df, interactions_df, events_df):
        self.friendships = friendships_df
        self.interactions = interactions_df
        self.events = events_df
        self._build_friendship_graph()
    
    def _build_friendship_graph(self):
        """Build adjacency set for O(1) friendship lookups."""
        self.friends = {}
        for _, row in self.friendships.iterrows():
            u1, u2 = row['user_id_1'], row['user_id_2']
            self.friends.setdefault(u1, set()).add(u2)
            self.friends.setdefault(u2, set()).add(u1)
    
    def get_friends(self, user_id):
        return self.friends.get(user_id, set())
    
    def compute(self, user_id, event_id):
        """Compute all social features for a (user, event) pair."""
        friends = self.get_friends(user_id)
        event = self.events[self.events['event_id'] == event_id].iloc[0]
        event_interactions = self.interactions[self.interactions['event_id'] == event_id]
        
        # 1. Total registered users
        registered_users = set(
            event_interactions[event_interactions['interaction_type'] == 'register']['user_id']
        )
        num_registered = len(registered_users)
        
        # 2. Registration ratio (registrations / impressions)
        num_impressions = len(
            event_interactions[event_interactions['interaction_type'] == 'impression']
        )
        registration_ratio = num_registered / max(num_impressions, 1)
        
        # 3. Friends registered
        friends_registered = friends & registered_users
        num_friends_registered = len(friends_registered)
        
        # 4. Friend registration ratio
        friend_reg_ratio = num_friends_registered / max(len(friends), 1)
        
        # 5. Invitations (from friends and non-friends)
        invites = event_interactions[
            (event_interactions['interaction_type'] == 'invite')
        ]
        # In our simplified data, we don't track invite targets,
        # but in production these would be tracked
        num_invites_from_friends = 0  # Simplified
        num_invites_from_others = 0   # Simplified
        
        # 6. Host is friend?
        host_is_friend = int(event['host_user_id'] in friends)
        
        # 7. Past events by this host that user attended
        host_events = set(self.events[self.events['host_user_id'] == event['host_user_id']]['event_id'])
        user_registrations = set(
            self.interactions[
                (self.interactions['user_id'] == user_id) & 
                (self.interactions['interaction_type'] == 'register')
            ]['event_id']
        )
        past_host_events_attended = len(host_events & user_registrations)
        
        return {
            'num_registered': num_registered,
            'registration_ratio': round(registration_ratio, 4),
            'num_friends_registered': num_friends_registered,
            'friend_registration_ratio': round(friend_reg_ratio, 4),
            'invites_from_friends': num_invites_from_friends,
            'invites_from_others': num_invites_from_others,
            'host_is_friend': host_is_friend,
            'past_host_events_attended': past_host_events_attended
        }


# Create sample data
friendships_df = pd.DataFrame({
    'user_id_1': [1, 1, 2, 3, 4],
    'user_id_2': [2, 5, 3, 5, 5],
})

events_df = pd.DataFrame({
    'event_id': [101, 102, 103],
    'host_user_id': [5, 3, 1],
})

interactions_df = pd.DataFrame({
    'user_id': [1, 2, 3, 5, 1, 2, 3, 5, 2, 5],
    'event_id': [101, 101, 101, 101, 101, 101, 101, 101, 101, 101],
    'interaction_type': [
        'impression', 'impression', 'impression', 'impression',
        'register', 'register', 'impression', 'register',
        'invite', 'invite'
    ]
})

computer = SocialFeatureComputer(friendships_df, interactions_df, events_df)

# Compute for different users and events
print("=== Social Features for Event 101 ===")
print(f"Registered users: {computer.compute(1, 101)['num_registered']}")
print()

for user_id in [1, 3, 4]:
    friends = computer.get_friends(user_id)
    features = computer.compute(user_id, 101)
    print(f"--- User {user_id} (friends: {friends}) ---")
    for feat, val in features.items():
        print(f"  {feat}: {val}")
    print()

## 5. Cold Start for New Events: Putting It All Together

In [ ]:
# === Cold Start Feature Availability Analysis ===

def analyze_cold_start_features():
    """
    Show which features are available at different stages of an event's lifecycle.
    
    This is a CRITICAL interview talking point.
    
    12-year-old version:
    When a new event is first posted, we cannot use features that
    depend on people having already seen it. But we CAN use features
    about the event itself and the user's preferences.
    """
    features = {
        # Feature: (available at creation?, available after some interactions?, category)
        "Same city": (True, True, "Location"),
        "Same country": (True, True, "Location"),
        "Distance bucket": (True, True, "Location"),
        "Walk score": (True, True, "Location"),
        "Walk score similarity": (True, True, "Location"),
        "Distance similarity": (True, True, "Location"),
        
        "Remaining time": (True, True, "Time"),
        "Remaining time similarity": (True, True, "Time"),
        "Day-of-week match": (True, True, "Time"),
        "Hour-of-day match": (True, True, "Time"),
        "Travel time": (True, True, "Time"),
        
        "Num registered": (False, True, "Social"),
        "Registration ratio": (False, True, "Social"),
        "Friends registered": (False, True, "Social"),
        "Host is friend": (True, True, "Social"),
        "Past host events attended": (True, True, "Social"),
        
        "Age bucket": (True, True, "User"),
        "Gender": (True, True, "User"),
        
        "Price bucket": (True, True, "Event"),
        "Price similarity": (True, True, "Event"),
        "Description similarity": (True, True, "Event"),
    }
    
    print("=== Feature Availability at Different Event Lifecycle Stages ===")
    print()
    print(f"{'Feature':<30} {'At Creation':>12} {'After Activity':>15} {'Category':>10}")
    print("-" * 75)
    
    available_at_creation = 0
    total_features = len(features)
    
    for feat, (at_creation, after_activity, category) in features.items():
        c = "YES" if at_creation else "NO"
        a = "YES" if after_activity else "NO"
        print(f"{feat:<30} {c:>12} {a:>15} {category:>10}")
        if at_creation:
            available_at_creation += 1
    
    print(f"\n=== Summary ===")
    print(f"Features available at event creation: {available_at_creation}/{total_features}")
    print(f"  ({available_at_creation/total_features*100:.0f}% -- we can recommend NEW events immediately!)")
    print(f"\nOnly {total_features - available_at_creation} features (registration-dependent social features)")
    print(f"require interaction history. These grow over time as the event gets exposure.")


analyze_cold_start_features()

In [ ]:
# === Batch vs Streaming Features ===

print("=== Batch vs Streaming Features (Interview Talking Point) ===")
print()

batch_features = {
    "User age": "Changes never/rarely",
    "User gender": "Changes never/rarely",
    "Event description": "Set once at creation",
    "Event price": "Set once at creation",
    "Event location": "Set once at creation",
    "Walk score": "Changes very rarely",
    "User temporal profile": "Updated daily/weekly",
    "User's avg past distance": "Updated daily/weekly",
    "Description similarity (TF-IDF)": "Updated daily/weekly",
}

streaming_features = {
    "Num users registered": "Changes with every registration",
    "Registration ratio": "Changes with every impression/registration",
    "Num friends registered": "Changes with every friend's registration",
    "Remaining time": "Changes every second!",
    "Estimated travel time": "Changes with traffic conditions",
}

print("BATCH (Static) Features -- computed periodically, stored in Feature Store:")
for feat, reason in batch_features.items():
    print(f"  - {feat}: {reason}")

print("\nSTREAMING (Dynamic) Features -- computed in real-time:")
for feat, reason in streaming_features.items():
    print(f"  - {feat}: {reason}")

print("\n=== Architecture Implication ===")
print("Batch features: stored in Feature Store (Redis, DynamoDB, etc.)")
print("  -> Computed by periodic batch jobs (Spark, Airflow)")
print("  -> Low latency reads at serving time")
print()
print("Streaming features: computed by real-time pipeline (Kafka, Flink)")
print("  -> Processed as events stream in")
print("  -> Sub-second freshness")

In [ ]:
# === Decay Factor for User Preferences ===

def compute_decayed_average(values, timestamps, current_time, half_life_days=30):
    """
    Compute a time-decayed average of user's historical values.
    
    12-year-old version:
    Your preferences change over time! If you loved hip-hop concerts
    6 months ago but recently started going to jazz events, we should
    weight your RECENT choices more. The 'decay factor' makes old
    preferences fade away gradually, like a fading memory.
    
    Staff-level detail:
    We use exponential decay: w(t) = exp(-lambda * age_days)
    where lambda = ln(2) / half_life_days.
    The 'half life' controls how fast old preferences fade.
    - Short half life (7 days) = only recent events matter
    - Long half life (90 days) = old events still contribute
    """
    decay_rate = np.log(2) / half_life_days
    
    weights = []
    for ts in timestamps:
        age_days = (current_time - ts).total_seconds() / 86400
        weight = np.exp(-decay_rate * age_days)
        weights.append(weight)
    
    weights = np.array(weights)
    values = np.array(values)
    
    return np.sum(values * weights) / np.sum(weights), weights


# Example: User's event distances over time
current_time = datetime(2024, 3, 15)

past_distances = [5, 8, 3, 45, 50, 55, 40, 2, 3, 5]  # miles
past_timestamps = [
    datetime(2023, 6, 1),  # 9 months ago (far events)
    datetime(2023, 7, 15),
    datetime(2023, 8, 20),
    datetime(2023, 9, 10),  # Started attending farther events
    datetime(2023, 10, 5),
    datetime(2023, 11, 15),
    datetime(2023, 12, 20),
    datetime(2024, 1, 10),  # Recently switched back to close events
    datetime(2024, 2, 14),
    datetime(2024, 3, 1),
]

# Compare simple average vs decayed average
simple_avg = np.mean(past_distances)
decayed_avg_short, weights_short = compute_decayed_average(
    past_distances, past_timestamps, current_time, half_life_days=30
)
decayed_avg_long, weights_long = compute_decayed_average(
    past_distances, past_timestamps, current_time, half_life_days=90
)

print("=== Decay Factor Example ===")
print(f"User's past event distances: {past_distances}")
print(f"\nSimple average distance: {simple_avg:.1f} miles")
print(f"  (Treats all past events equally)")
print(f"\nDecayed average (half-life=30 days): {decayed_avg_short:.1f} miles")
print(f"  (Recent events dominate -- captures switch to nearby events)")
print(f"\nDecayed average (half-life=90 days): {decayed_avg_long:.1f} miles")
print(f"  (More balanced between old and new preferences)")

print(f"\nWeights with 30-day half-life (most recent first):")
for i, (d, ts, w) in enumerate(zip(past_distances, past_timestamps, weights_short)):
    age = (current_time - ts).days
    bar = '#' * int(w * 30)
    print(f"  {age:>4} days ago: {d:>3} miles, weight={w:.4f} {bar}")

## Key Takeaways

### Location Features
1. Walk score + transit score + bike score capture venue accessibility
2. Distance buckets handle the "how far" question
3. SIMILARITY features compare to user's historical preferences (not just raw values)
4. Efficiency trade-off: pre-compute distances vs. pass raw coordinates to model

### Time Features
1. Remaining time matters differently for different users (planners vs. spontaneous)
2. Day-of-week and hour-of-day profiles capture schedule preferences
3. Travel time + estimated arrival gives feasibility signal

### Social Features
1. Friends attending is one of the strongest signals
2. Host relationship matters (friend? past events attended?)
3. Invitations from friends are powerful signals

### Cold Start
1. ~85% of features are available at event creation (no interactions needed)
2. Content-based features (description, category, price) enable immediate recommendations
3. Interaction-dependent features (registration count) grow over event lifecycle
4. Feature engineering is the primary tool to combat cold start

### Engineering Best Practices
1. Batch vs. streaming features split for scalability
2. Decay factor for evolving user preferences
3. Feature store for static features, real-time computation for dynamic ones

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)